# 02 — Preparación de datos
**Proyecto:** Recomendador de draft de Brawl Stars con reconocimiento visual de brawlers

**Objetivo:** convertir las partidas crudas de `EliF77/brawlstars-ranked` en una tabla **larga** con una fila por brawler por partida (con su equipo, mapa, modo y si ganó), filtrarla por mapa y modo, y elegir los 20 brawlers del alcance inicial según su número de partidas.

**Antes de ejecutar:** revisa la celda 4 (detección de columnas) y confirma los nombres con la salida del notebook 01. Si la detección automática se equivoca, corrígela en la celda 5. El notebook se detiene con un mensaje claro si algo no cuadra; no adivina en silencio.

## 1. Configuración

In [5]:
DATASET_ID = "EliF77/brawlstars-ranked"
LIMIT_ROWS = 300_000     # filas a usar; None = dataset completo (revisa la RAM antes)
MIN_MATCHES = 100        # mínimo de partidas para que una estadística sea confiable
N_BRAWLERS = 20          # alcance inicial
MAP_FILTER = None        # p. ej. "Gem Grab"; None = todos los mapas
MODE_FILTER = None       # p. ej. "Ranked"; None = todos los modos
OUT_DIR = "/content/data"

## 2. Imports y carga
Cargamos el dataset completo en memoria (solo las primeras `LIMIT_ROWS` filas si se define el límite) y mostramos su forma y su uso de memoria.

In [6]:
import os, time
import numpy as np
import pandas as pd
from datasets import load_dataset

os.makedirs(OUT_DIR, exist_ok=True)

t0 = time.time()
ds = load_dataset(DATASET_ID, split="train")
df_raw = ds.to_pandas()
if LIMIT_ROWS:
    df_raw = df_raw.head(LIMIT_ROWS).reset_index(drop=True)

print("Forma:", df_raw.shape)
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Tiempo de carga: {time.time() - t0:.0f} s")
df_raw.head(3)

Forma: (300000, 36)
Memoria: 0.35 GB
Tiempo de carga: 9 s


,id,battle_time,mode,map,record,star_brawler,star_power,star_player_tag,star_elo,avg_elo,...,t2_b1_name,t2_b1_elo,t2_b1_power,t2_b1_tag,t2_b2_name,t2_b2_elo,t2_b2_power,t2_b2_tag,skill_ns,skill_ns_ok
0,71079,20261119T090001.000Z,heist,Kaboom Canyon,T2-T2,MELODIE,11,#2LV0VPJ0VG,16,15.833333,...,COLT,16,11,#9PL2GQ2LR,MELODIE,16,11,#2LV0VPJ0VG,1.392557,1
1,80590,20261119T090001.000Z,brawlBall,Triple Dribble,T1-T2-T1,RICO,11,#PR2VGG9Q9,15,14.833333,...,SHADE,15,11,#P0J8QP2UU,WILLOW,14,11,#928JJPU2,0.857005,1
2,170711,20261119T090001.000Z,hotZone,Dueling Beetles,T1-T1,KENJI,11,#2G9992GG9R,14,13.166667,...,AMBER,12,11,#2LC2GJ0PJC,JACKY,14,11,#JQUY2GCL,-0.338268,1


## 3. Detección de columnas
Buscamos columnas candidatas por palabras clave: mapa, modo, resultado y las columnas de equipos o brawlers. Es una heurística; por eso la celda siguiente permite corregirla.

In [3]:
def find_cols(cols, keywords):
    return [c for c in cols if any(k in c.lower() for k in keywords)]

guess = {
    "map": find_cols(df_raw.columns, ["map", "mapa"]),
    "mode": find_cols(df_raw.columns, ["mode", "modo"]),
    "result": find_cols(df_raw.columns, ["result", "win", "winner", "victor", "outcome"]),
    "team": find_cols(df_raw.columns, ["team", "equipo", "brawler", "pick"]),
}
for k, v in guess.items():
    print(f"{k:>8}: {v}")

print("\nTipos y ejemplos:")
for c in df_raw.columns:
    print(f"- {c} | {df_raw[c].dtype} | ej: {repr(df_raw[c].iloc[0])[:80]}")

     map: ['map']
    mode: ['mode']
  result: []
    team: ['star_brawler']

Tipos y ejemplos:
- id | int64 | ej: np.int64(71079)
- battle_time | object | ej: '20261119T090001.000Z'
- mode | object | ej: 'heist'
- map | object | ej: 'Kaboom Canyon'
- record | object | ej: 'T2-T2'
- star_brawler | object | ej: 'MELODIE'
- star_power | int64 | ej: np.int64(11)
- star_player_tag | object | ej: '#2LV0VPJ0VG'
- star_elo | int64 | ej: np.int64(16)
- avg_elo | float64 | ej: np.float64(15.833333333333334)
- t1_b0_name | object | ej: 'AMBER'
- t1_b0_elo | int64 | ej: np.int64(16)
- t1_b0_power | int64 | ej: np.int64(11)
- t1_b0_tag | object | ej: '#9G9CYJP98'
- t1_b1_name | object | ej: 'BROCK'
- t1_b1_elo | int64 | ej: np.int64(16)
- t1_b1_power | int64 | ej: np.int64(11)
- t1_b1_tag | object | ej: '#2LYYLQG228'
- t1_b2_name | object | ej: 'KAZE'
- t1_b2_elo | int64 | ej: np.int64(15)
- t1_b2_power | int64 | ej: np.int64(11)
- t1_b2_tag | object | ej: '#R2JGLLRRC'
- t2_b0_name | object | ej: 

## 4. Confirmación de columnas
Aquí fijas los nombres definitivos. Si lo dejas en `None`, se usa la primera candidata de la detección. Después verificamos que todas existan.

- `team_cols`: lista de columnas de equipos. Pueden ser dos columnas con listas de brawlers (una por equipo) o columnas con un brawler por celda.
- `result_col`: columna del resultado.
- `win_rule`: cómo interpretar el resultado. `"winner_index"` si el valor indica el índice del equipo ganador (0 o 1, o 1 o 2 según el dataset). `"per_team_bool"` si cada equipo tiene su propio valor de victoria/derrota.

In [8]:
# Agrupamos los brawlers de cada equipo en listas para poder usar la lógica del notebook
df_raw["team_a"] = df_raw[["t1_b0_name", "t1_b1_name", "t1_b2_name"]].values.tolist()
df_raw["team_b"] = df_raw[["t2_b0_name", "t2_b1_name", "t2_b2_name"]].values.tolist()

CONFIRMED = {
    "map": "map",
    "mode": "mode",
    "result": "record",
    "team_cols": ["team_a", "team_b"],
    "win_rule": "per_team_bool",    # Puedes ajustarlo según cómo interpretes 'record'
    "winner_base": 0,
}

map_col = CONFIRMED["map"] or (guess["map"][0] if guess["map"] else None)
mode_col = CONFIRMED["mode"] or (guess["mode"][0] if guess["mode"] else None)
result_col = CONFIRMED["result"] or (guess["result"][0] if guess["result"] else None)
team_cols = CONFIRMED["team_cols"] or guess["team"][:2]

for name, col in [("map", map_col), ("mode", mode_col), ("result", result_col)]:
    assert col is not None and col in df_raw.columns, f"No se encontró la columna de {name}; revisa la celda 3"
assert len(team_cols) == 2 and all(c in df_raw.columns for c in team_cols), \
    f"Se necesitan exactamente dos columnas de equipos; encontradas: {team_cols}"

print("Columnas usadas:")
print("  mapa:", map_col)
print("  modo:", mode_col)
print("  resultado:", result_col)
print("  equipos:", team_cols)
print("\nValores del resultado:", df_raw[result_col].astype(str).value_counts().head(10).to_dict())

Columnas usadas:
  mapa: map
  modo: mode
  resultado: record
  equipos: ['team_a', 'team_b']

Valores del resultado: {'T1-T1': 87516, 'T2-T2': 84890, 'T2-T1-T1': 24490, 'T1-T2-T2': 24400, 'T1-T2-T1': 23425, 'T2-T1-T2': 23323, 'T1': 15053, 'T2': 14911, 'D-T1-T1': 267, 'D-T2-T2': 264}


## 5. Construcción de la tabla larga
Pasamos de una fila por partida a una fila por **brawler por partida**. Si las columnas de equipos contienen listas, cada elemento de la lista se convierte en una fila. Si contienen un solo brawler por celda, se conserva ese valor.

Después calculamos la columna `win` (1 si el brawler estuvo en el equipo ganador, 0 si no) según la regla confirmada.

In [9]:
base = df_raw.copy()
base["match_id"] = np.arange(len(base))

parts = []
for side, col in enumerate(team_cols):
    sub = base[["match_id", map_col, mode_col, result_col]].copy()
    sub["side"] = side
    sub["brawler"] = base[col]
    parts.append(sub)

long = pd.concat(parts, ignore_index=True)

is_list = long["brawler"].map(lambda x: isinstance(x, (list, tuple, np.ndarray))).any()
if is_list:
    long = long.explode("brawler", ignore_index=True)

long = long.dropna(subset=["brawler"]).reset_index(drop=True)
print("Filas en tabla larga:", len(long))
print("Columnas formato lista:", is_list)
long.head(5)

Filas en tabla larga: 1800000
Columnas formato lista: True


,match_id,map,mode,record,side,brawler
0,0,Kaboom Canyon,heist,T2-T2,0,AMBER
1,0,Kaboom Canyon,heist,T2-T2,0,BROCK
2,0,Kaboom Canyon,heist,T2-T2,0,KAZE
3,1,Triple Dribble,brawlBall,T1-T2-T1,0,RICO
4,1,Triple Dribble,brawlBall,T1-T2-T1,0,BARLEY


Ahora calculamos `win`. Si la regla no está confirmada, el notebook se detiene para que la definas.

In [10]:
rule = CONFIRMED["win_rule"]
assert rule in ("winner_index", "per_team_bool"), \
    "Define CONFIRMED['win_rule'] en la celda 4 antes de continuar (revisa los valores del resultado)."

if rule == "winner_index":
    winner_side = long[result_col].astype(int) - CONFIRMED["winner_base"]
    long["win"] = (winner_side == long["side"]).astype(int)
else:
    # Cada equipo tiene su propio resultado: 1/True = victoria
    long["win"] = long[result_col].map(lambda v: 1 if str(v).lower() in ("1", "true", "win", "victory") else 0)

print("Distribución de win:", long["win"].value_counts().to_dict())

Distribución de win: {0: 1800000}


## 6. Validación de integridad
Antes de usar los datos comprobamos tres cosas: que cada partida tenga exactamente un equipo ganador, que el número de brawlers por equipo sea el esperado y que no haya registros vacíos.

In [12]:
# 1. Corregimos el cálculo de victoria analizando la columna 'record'
# Si el último set en 'record' termina en 'T1', ganó el Equipo 1 (side = 0).
# Si termina en 'T2', ganó el Equipo 2 (side = 1).
def determinar_ganador(record_val):
    rec_str = str(record_val).strip().upper()
    if rec_str.endswith("T1"):
        return 0  # Lado 0 (team_a)
    elif rec_str.endswith("T2"):
        return 1  # Lado 1 (team_b)
    return -1     # Indefinido / Empate

# Aplicamos la lógica y recalculamos 'win'
winner_side = long["record"].map(determinar_ganador)
long["win"] = (winner_side == long["side"]).astype(int)

# 2. Procedemos con la validación de integridad
winners_per_match = long.groupby(["match_id", "side"])["win"].max().groupby("match_id").sum()
bad_matches = (winners_per_match != 1).sum()
print(f"Partidas sin exactamente un ganador: {bad_matches} de {len(winners_per_match)}")

size_per_team = long.groupby(["match_id", "side"]).size()
print("Tamaño de equipo más frecuente:", size_per_team.value_counts().head(5).to_dict())

print("Brawlers vacíos:", long["brawler"].isna().sum())
print("Brawlers únicos:", long["brawler"].nunique())

assert bad_matches / max(len(winners_per_match), 1) < 0.01, \
    "Más del 1 % de partidas sin ganador claro: revisa win_rule antes de continuar."
print("Validación aprobada con éxito.")

Partidas sin exactamente un ganador: 397 de 300000
Tamaño de equipo más frecuente: {3: 600000}
Brawlers vacíos: 0
Brawlers únicos: 108
Validación aprobada con éxito.


## 7. Filtros de mapa y modo
Filtramos según `MAP_FILTER` y `MODE_FILTER`. Antes mostramos los valores disponibles para elegir bien.

In [13]:
print("Mapas (top 10):", long[map_col].value_counts().head(10).to_dict())
print("Modos (top 10):", long[mode_col].value_counts().head(10).to_dict())

filtered = long
if MAP_FILTER is not None:
    filtered = filtered[filtered[map_col] == MAP_FILTER]
if MODE_FILTER is not None:
    filtered = filtered[filtered[mode_col] == MODE_FILTER]

print(f"\nFilas tras filtros: {len(filtered)}")

Mapas (top 10): {'Out in the Open': 77658, 'New Horizons': 77304, "Belle's Rock": 76488, 'Dry Season': 76434, 'Shooting Star': 76350, 'Hideout': 76308, 'Pinball Dreams': 76110, 'Sneaky Fields': 76104, 'Center Stage': 75486, 'Flaring Phoenix': 75390}
Modos (top 10): {'knockout': 306840, 'bounty': 303000, 'brawlBall': 302724, 'heist': 296718, 'hotZone': 295692, 'gemGrab': 295026}

Filas tras filtros: 1800000


## 8. Selección de los 20 brawlers
Elegimos los `N_BRAWLERS` brawlers con más apariciones en la muestra filtrada. Así cada clase tiene soporte estadístico suficiente. Esta es la justificación que se documenta en el informe.

In [14]:
counts = filtered["brawler"].value_counts()
top = counts.head(N_BRAWLERS)
print(f"Brawlers disponibles: {len(counts)}")
print(f"Apariciones del brawler nº {N_BRAWLERS}: {top.iloc[-1]}")
print("\nSelección:")
print(top.to_string())

selected = top.index.tolist()
long_sel = filtered[filtered["brawler"].isin(selected)].copy()
print(f"\nFilas con los brawlers seleccionados: {len(long_sel)}")

Brawlers disponibles: 108
Apariciones del brawler nº 20: 29164

Selección:
brawler
AMBER      85578
BROCK      83345
SPIKE      69681
RICO       67600
GRIFF      57731
EDGAR      50530
SURGE      45043
COLETTE    44270
BO         42645
COLT       41757
MANDY      41181
MAX        40867
PIPER      40325
MORTIS     38894
EMZ        38674
CHESTER    37528
MICO       33523
GUS        32484
MR. P      30731
MELODIE    29164

Filas con los brawlers seleccionados: 951551


## 9. Guardado
Guardamos la tabla larga completa, la de los brawlers seleccionados y la lista de brawlers. El notebook 03 usará estos archivos.

In [15]:
long.to_parquet(f"{OUT_DIR}/brawl_long.parquet", index=False)
long_sel.to_parquet(f"{OUT_DIR}/brawl_selected.parquet", index=False)
pd.Series(selected, name="brawler").to_csv(f"{OUT_DIR}/selected_brawlers.csv", index=False)

resumen = {
    "partidas_originales": len(df_raw),
    "filas_tabla_larga": len(long),
    "filas_filtradas": len(filtered),
    "brawlers_disponibles": int(len(counts)),
    "brawlers_seleccionados": len(selected),
    "filas_seleccionadas": len(long_sel),
    "min_matches_configurado": MIN_MATCHES,
}
for k, v in resumen.items():
    print(f"{k}: {v}")
print("\nArchivos guardados en", OUT_DIR)
print(os.listdir(OUT_DIR))

partidas_originales: 300000
filas_tabla_larga: 1800000
filas_filtradas: 1800000
brawlers_disponibles: 108
brawlers_seleccionados: 20
filas_seleccionadas: 951551
min_matches_configurado: 100

Archivos guardados en /content/data
['brawl_long.parquet', 'brawl_selected.parquet', 'selected_brawlers.csv']
